# Step 7 — Adversarial Persona Simulator
**Tough Talks · Phase 3**

Goal: prove Gemma 4 E2B (text-only) can roleplay a recurring counterparty across a multi-turn practice conversation, staying realistically resistant based on a PersonVault profile from Step 06.

Each persona reply is one structured JSON object matching `data/schemas/persona_reply.schema.json`:

- `persona_name` — forced to the configured persona name in code (system-contract enforcement, not trusted to the model).
- `reply` — what the persona actually says, in character.
- `resistance_type` — closed enum (`deflect | guilt_trip | deny | counter_attack | silent | concede`). Synonym normaliser handles model vocabulary drift.
- `escalation_level` — clamped to `[0, 1]`.

**Architecture** — same hybrid-runtime pattern as Steps 04 / 05 / 06:
- Runtime in `backend/core/_runtime/persona_sim.py`. Notebook is a thin driver.
- Rolling history: each prior persona reply is fed back as a JSON-serialised `role=assistant` turn, so the model sees its own escalation arc and resistance compounds across turns.
- Two-shot retry: greedy first, then a single light-sampling pass (`temperature=0.3, top_p=0.9, top_k=64`) on JSON / validation failure.
- `enable_thinking` is exposed as a knob, defaulting to `False`. The final cell A/B-compares thinking-on vs thinking-off on the same first user turn to test the open hypothesis in `knowledge/phases/hypotheses.md`.

**Pronoun discipline (carried into the prompt)** — the prompt explicitly states that `emotional_triggers` are USER-side cues that escalate YOU (the persona) and `de_escalation_keys` are USER-side moves that calm YOU, with off-transcript examples. Same lesson as Step 06's trigger-side fix in `knowledge/phases/rules.md`.

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `generate_persona_reply()` returns a schema-conforming dict for a cold-start turn (no history).
3. `run_practice_conversation()` walks a 5-turn user script and returns alternating user / persona entries. The persona's `escalation_level` trajectory tracks the user's pressure / de-escalation, not a flat baseline.
4. `resistance_type` stays inside the enum across all turns (synonym normaliser unused or used silently — both are fine).
5. Each persona reply validates against `data/schemas/persona_reply.schema.json` (required fields, resistance enum, escalation bounds).
6. The A/B cell renders both `enable_thinking=False` and `enable_thinking=True` outputs side-by-side for the same first user turn — qualitative judgement for Solmaz, not a hard PASS / FAIL.

**Note on the input profile.** The PersonVault profile for Jamie is hardcoded inline (matching the v2 output Step 06 produced) so this notebook doesn't re-pay Step 06's ~15-minute LLM cost on every run. In production this dict comes from `analyze_person_vault()`.

In [1]:
# ── 0. Install / upgrade dependencies ───────────────────────────────
# Text-only path — no audio libs required. Same rule as Steps 05 / 06:
# bump only transformers + accelerate on Colab / Kaggle (bumping torch
# breaks the pre-installed torchvision / CUDA pairing). After this
# first run, RESTART THE KERNEL before continuing if you actually
# upgraded transformers — the already-imported version won't pick up
# the change.

!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 131.6 MB/s eta 0:00:0000:010:01


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────
# Same shim as Steps 01–06 — auto-clones / refreshes on Colab / Kaggle
# and clears any cached `backend.*` modules so the imports below pick
# up the freshly-pulled code instead of whatever this kernel imported
# earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ───────────────────────────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    ALLOWED_RESISTANCE_TYPES,
    DEFAULT_MODEL_ID,
    LoadConfig,
    PersonaReplyError,
    PersonaSimConfig,
    format_persona_profile_block,
    generate_persona_reply,
    load_model,
    run_practice_conversation,
)

In [4]:
# ── 3. Configuration ─────────────────────────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "persona_reply.schema.json"

# PersonVault profile for Jamie. This matches the v2 output Step 06
# produced after Conversation A + Conversation B (defensive in A,
# softened to assertive in B, accumulated triggers / de-escalation
# keys / deflections across both conversations). Inlined here so this
# notebook doesn't re-pay Step 06's ~15-minute LLM cost on every run.
# In production this dict comes from `analyze_person_vault()`.
JAMIE_PROFILE = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": [
            "citing past commitments",
            "suggesting escalation when blockers are still open",
            "implying missed communication is one-sided",
            "setting hard deadlines without acknowledging blockers",
        ],
        "de_escalation_keys": [
            "explicitly disowning blame",
            "reframing as joint problem-solving",
            "acknowledging the tightness of the timeline",
            "proposing a concrete next-step the user will own",
        ],
        "common_deflections": [
            "I told you",
            "Don't blame me",
            "Don't put this on me",
            "You weren't there",
            "staging tables aren't done",
        ],
        "responds_best_to": (
            "Clear, actionable next steps tied to specific milestones. "
            "Responds well when the user accepts responsibility for "
            "communication gaps."
        ),
    },
}

# What the USER wants to achieve in this practice round. The persona
# will push back against this — they don't share the user's goal.
USER_GOAL = (
    "Get Jamie to commit to delivering the staging-tables data by "
    "Wednesday EOD and to acknowledge that the escalation last week "
    "needed to land more clearly — not just be sent."
)

# Five user turns that walk the practice conversation through an
# escalation → de-escalation arc. The expected persona behaviour:
# - Turns 1–2: triggers fire → deflect / counter_attack with high escalation.
# - Turn 3: USER disowns blame and reframes as joint problem-solving — a
#   de_escalation_key — so Jamie should partially soften.
# - Turn 4: concrete next-step proposal that matches `responds_best_to`
#   — Jamie should move toward concede.
# - Turn 5: mutual accountability close — Jamie should concede.
USER_MESSAGES = [
    "Hey Jamie, I want to talk about what happened with the report "
    "last week. The fallout from missing that deadline is still on us, "
    "and I'd like to make sure it doesn't happen again.",
    "I hear that the data team hadn't delivered, but the report date "
    "matters more than who knew what on which day. Can we agree the "
    "staging data lands by Wednesday EOD this time?",
    "I'm not saying you didn't escalate. I'm saying the escalation has "
    "to land — if I miss your emails, that's still our problem to "
    "solve together, not yours alone.",
    "What if we set up a five-minute Slack check-in each morning until "
    "the staging tables are done? You flag what's blocked, I unblock "
    "it that day. That way nothing has to escalate at all.",
    "Thank you, that means a lot. And next time something blocks the "
    "report, can you ping me directly instead of email? I'll do the "
    "same when I'm the one missing things.",
]

print(f"Model              : {MODEL_ID}")
print(f"Device             : {DEVICE}")
print(f"Persona            : {JAMIE_PROFILE['name']} ({JAMIE_PROFILE['relationship_type']})")
print(f"Resistance enum    : {ALLOWED_RESISTANCE_TYPES}")
print(f"User turns planned : {len(USER_MESSAGES)}")

Model              : google/gemma-4-E2B-it
Device             : cuda
Persona            : Jamie (colleague)
Resistance enum    : ('deflect', 'guilt_trip', 'deny', 'counter_attack', 'silent', 'concede')
User turns planned : 5


In [5]:
# ── 4. Preview the rendered profile block (no model needed) ──────────────
# `format_persona_profile_block` is what the runtime feeds into the
# system prompt. Rendering it here is purely diagnostic — it lets us
# verify the profile is well-formed before paying for the model load.

print(format_persona_profile_block(JAMIE_PROFILE))

- communication_style: defensive
- emotional_triggers (USER-side cues that escalate / make you defensive): ['citing past commitments', 'suggesting escalation when blockers are still open', 'implying missed communication is one-sided', 'setting hard deadlines without acknowledging blockers']
- de_escalation_keys (USER-side moves that calm / open you up): ['explicitly disowning blame', 'reframing as joint problem-solving', 'acknowledging the tightness of the timeline', 'proposing a concrete next-step the user will own']
- common_deflections (YOUR habitual evasion phrases): ['I told you', "Don't blame me", "Don't put this on me", "You weren't there", "staging tables aren't done"]
- responds_best_to: Clear, actionable next steps tied to specific milestones. Responds well when the user accepts responsibility for communication gaps.


In [6]:
# ── 5. Load the text-only processor + model ─────────────────────────
# Persona simulation reasons over WORDS — same rationale as TalkDNA /
# PersonVault. The `multimodal=False` (default) path loads
# `AutoModelForCausalLM`, which is lighter on VRAM and slightly faster
# than the multimodal class. Live Mode (Phase 5+) will reuse this same
# loaded model across components, so the load cost is amortised.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded (5.1B parameters, on cuda:0)


In [7]:
# ── 6. Run the multi-turn practice conversation (greedy, thinking=False) ───
# `run_practice_conversation` calls `generate_persona_reply` once per
# user message, threading the rolling history automatically. Each
# persona reply is a schema-conforming dict; we print them as we go so
# the escalation arc is visible turn-by-turn rather than only in the
# final results table.

cfg = PersonaSimConfig(
    persona_profile=JAMIE_PROFILE,
    user_goal=USER_GOAL,
    enable_thinking=False,
)

history: list[dict] = []
persona_turns: list[dict] = []
for i, user_msg in enumerate(USER_MESSAGES, start=1):
    try:
        reply = generate_persona_reply(
            processor, model, user_msg, history=history, cfg=cfg
        )
    except PersonaReplyError as exc:
        print(f"\n[Turn {i}] FAILED: {exc}")
        for attempt in getattr(exc, "attempts", []):
            print(f"  attempt ({attempt['sampling']}): {attempt['error']}")
            print(f"  raw_head: {attempt['raw_head']!r}")
        raise
    history.append({"speaker": "user", "text": user_msg})
    history.append({"speaker": "persona", **reply})
    persona_turns.append(reply)

    print(f"\n=== Turn {i} ===")
    print(f"USER  : {user_msg}")
    print(f"JAMIE : {reply['reply']}")
    print(f"        resistance_type = {reply['resistance_type']}")
    print(f"        escalation_level = {reply['escalation_level']:.2f}")


=== Turn 1 ===
USER  : Hey Jamie, I want to talk about what happened with the report last week. The fallout from missing that deadline is still on us, and I'd like to make sure it doesn't happen again.
JAMIE : Look, I told you the blockers were there. We can't just expect things to happen perfectly on schedule every time.
        resistance_type = deflect
        escalation_level = 0.50

=== Turn 2 ===
USER  : I hear that the data team hadn't delivered, but the report date matters more than who knew what on which day. Can we agree the staging data lands by Wednesday EOD this time?
JAMIE : I told you, staging tables aren't done yet. Wednesday EOD is ambitious, but fine, I'll push for it.
        resistance_type = silent
        escalation_level = 0.60

=== Turn 3 ===
USER  : I'm not saying you didn't escalate. I'm saying the escalation has to land — if I miss your emails, that's still our problem to solve together, not yours alone.
JAMIE : Don't put this on me. I'm just doing my job. I

In [8]:
# ── 7. Schema validation + results table ──────────────────────────────
# Hand-rolled validator (no jsonschema dep — matches Steps 04 / 05 / 06).
# Per turn:
#   - required schema fields present (persona_name, reply,
#     resistance_type, escalation_level)
#   - resistance_type in the schema enum
#   - escalation_level in [0, 1]
#   - persona_name matches the configured name (system-contract check)
#   - reply is a non-empty string
# Cross-turn:
#   - turn_id values are unique (runtime-attached uuids)
#   - escalation arc shows variance (not a flat baseline)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
SCHEMA_REQUIRED  = schema["required"]
RESISTANCE_ENUM  = schema["properties"]["resistance_type"]["enum"]


def _validate_persona_reply(reply: dict) -> list[str]:
    errs: list[str] = []
    for k in SCHEMA_REQUIRED:
        if k not in reply:
            errs.append(f"missing required key {k!r}")
    rt = reply.get("resistance_type")
    if rt is not None and rt not in RESISTANCE_ENUM:
        errs.append(f"resistance_type {rt!r} not in {RESISTANCE_ENUM}")
    esc = reply.get("escalation_level")
    if isinstance(esc, (int, float)):
        if not (0.0 <= float(esc) <= 1.0):
            errs.append(f"escalation_level {esc} outside [0, 1]")
    elif esc is not None:
        errs.append(f"escalation_level not numeric: {esc!r}")
    name = reply.get("persona_name")
    if not isinstance(name, str) or not name.strip():
        errs.append(f"persona_name empty or non-string: {name!r}")
    text = reply.get("reply")
    if not isinstance(text, str) or not text.strip():
        errs.append(f"reply empty or non-string: {text!r}")
    return errs


per_turn_errors = [_validate_persona_reply(r) for r in persona_turns]
n_clean = sum(1 for errs in per_turn_errors if not errs)

turn_ids = [r.get("turn_id") for r in persona_turns]
ids_unique = len(set(turn_ids)) == len(turn_ids)

escalations = [r.get("escalation_level", 0.0) for r in persona_turns]
escalation_variance = (max(escalations) - min(escalations)) if escalations else 0.0

name_correct = all(
    r.get("persona_name") == JAMIE_PROFILE["name"] for r in persona_turns
)

checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",       True,                                  f"{n_params:.1f}B params on {model.device}"),
    ("all_turns_returned",      len(persona_turns) == len(USER_MESSAGES), f"{len(persona_turns)}/{len(USER_MESSAGES)}"),
    ("persona_name_forced",     name_correct,                          f"all turns -> {JAMIE_PROFILE['name']}"),
    ("schema_valid_per_turn",   n_clean == len(persona_turns),         f"{n_clean}/{len(persona_turns)} clean"),
    ("resistance_in_enum",      all(r.get("resistance_type") in RESISTANCE_ENUM for r in persona_turns),
                                                                       f"{[r.get('resistance_type') for r in persona_turns]}"),
    ("escalation_in_bounds",    all(0.0 <= float(r.get("escalation_level", 0.0)) <= 1.0 for r in persona_turns),
                                                                       f"{[round(e, 2) for e in escalations]}"),
    ("turn_ids_unique",         ids_unique,                            f"{len(set(turn_ids))} unique / {len(turn_ids)} total"),
    ("escalation_has_variance", escalation_variance > 0.05,            f"max - min = {escalation_variance:.2f}"),
]

print("=" * 76)
print("STEP 7 RESULTS — Gemma 4 Adversarial Persona Simulator")
print("=" * 76)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

print("\nPersona turn trajectory:")
for i, r in enumerate(persona_turns, start=1):
    print(f"  Turn {i}: resistance={r['resistance_type']:<14s} "
          f"escalation={r['escalation_level']:.2f}")
    print(f"          reply: {r['reply']}")

if any(errs for errs in per_turn_errors):
    print("\nSchema errors:")
    for i, errs in enumerate(per_turn_errors, start=1):
        if not errs:
            continue
        print(f"  Turn {i}:")
        for line in errs:
            print(f"    - {line}")

print()
print("OVERALL:", "READY FOR STEP 8" if all_ok else "FIX FAILURES ABOVE")

STEP 7 RESULTS — Gemma 4 Adversarial Persona Simulator
[PASS]  text_model_loaded             5.1B params on cuda:0
[PASS]  all_turns_returned            5/5
[PASS]  persona_name_forced           all turns -> Jamie
[PASS]  schema_valid_per_turn         5/5 clean
[PASS]  resistance_in_enum            ['deflect', 'silent', 'deny', 'concede', 'concede']
[PASS]  escalation_in_bounds          [0.5, 0.6, 0.7, 0.4, 0.3]
[PASS]  turn_ids_unique               5 unique / 5 total
[PASS]  escalation_has_variance       max - min = 0.40

Persona turn trajectory:
  Turn 1: resistance=deflect        escalation=0.50
          reply: Look, I told you the blockers were there. We can't just expect things to happen perfectly on schedule every time.
  Turn 2: resistance=silent         escalation=0.60
          reply: I told you, staging tables aren't done yet. Wednesday EOD is ambitious, but fine, I'll push for it.
  Turn 3: resistance=deny           escalation=0.70
          reply: Don't put this on me. I'm

In [9]:
# ── 8. A/B: enable_thinking=False vs enable_thinking=True (cold start) ─────
# Tests the open hypothesis from `knowledge/phases/hypotheses.md` —
# does Gemma 4's thinking channel help the model reason about the
# `resistance_type` / `escalation_level` choice before producing the
# in-character reply? Same first user message, no history, otherwise
# identical config. Qualitative side-by-side render; Solmaz decides
# whether to promote / demote the hypothesis.

first_user_msg = USER_MESSAGES[0]

cfg_no_thinking = PersonaSimConfig(
    persona_profile=JAMIE_PROFILE,
    user_goal=USER_GOAL,
    enable_thinking=False,
)
cfg_thinking = PersonaSimConfig(
    persona_profile=JAMIE_PROFILE,
    user_goal=USER_GOAL,
    enable_thinking=True,
    # Thinking burns more tokens before the JSON — give it headroom.
    max_new_tokens=768,
)

reply_no_thinking = generate_persona_reply(
    processor, model, first_user_msg, history=[], cfg=cfg_no_thinking
)
reply_thinking = generate_persona_reply(
    processor, model, first_user_msg, history=[], cfg=cfg_thinking
)

print("=" * 76)
print("A/B — enable_thinking on the same cold-start turn")
print("=" * 76)
print(f"USER: {first_user_msg}\n")

for label, r in (("thinking=False (default)", reply_no_thinking),
                  ("thinking=True",            reply_thinking)):
    print(f"--- {label} ---")
    print(f"  reply           : {r['reply']}")
    print(f"  resistance_type : {r['resistance_type']}")
    print(f"  escalation_level: {r['escalation_level']:.2f}")
    print()

A/B — enable_thinking on the same cold-start turn
USER: Hey Jamie, I want to talk about what happened with the report last week. The fallout from missing that deadline is still on us, and I'd like to make sure it doesn't happen again.

--- thinking=False (default) ---
  reply           : Look, I told you the blockers were there. We can't just expect things to happen perfectly on schedule every time.
  resistance_type : deflect
  escalation_level: 0.50

--- thinking=True ---
  reply           : We both know things got complicated. It wasn't just one person's fault.
  resistance_type : deny
  escalation_level: 0.50

